This notebook runs what is needed to create renderings of the MDNs in the brain and all their pre- and postsynaptic sites (Fig. 1j, 2a; Extended Data Fig. 1b; Supplemental Data Synapse Topology) and of the Bolt neurons (Extended Data Fig. 1e)

Use the `FAFB` environment defined by `/envs/ENV_FAFB.yml` to run this notebook. See the readme in the envs folder for installation instructions.

authored by https://github.com/nmancini31

In [ ]:
from fafbseg import flywire

In [ ]:
# https://fafbseg-py.readthedocs.io/en/stable/source/tutorials/flywire_setup.html for information on api key. Paste yours below or in ./data/fafb_api_key.txt 
api_key = "your api key here"
if api_key == "your api key here":
    with open("../data/fafb_api_key.txt", "r", encoding="utf-8") as f:
        api_key = f.read().strip()
flywire.set_chunkedgraph_secret(api_key, overwrite=True)

In [ ]:
## code to display neuron of interest + presynapses from specific input neurons

import numpy as np
import datetime
import pandas as pd
from caveclient import CAVEclient

datastack_name = "flywire_fafb_production"
client = CAVEclient(datastack_name)

# Define the segments of interest for the pre-synaptic neurons
pre_synaptic_neurons = [720575940607800242] #e.g. LAL106
    
# Define the segments of interest for the postsynaptic neurons
post_synaptic_neurons = [720575940627652358] #e.g. MDN

# Query synapses from above cells. Here I am querying the output synapses
# Note: # # Query_view() has filter: syn cleft score > 50 and syn within 100nm radius is merged
syn_df = client.materialize.query_view("valid_synapses_nt_np", 
                                    filter_in_dict={"pre_pt_root_id": pre_synaptic_neurons, "post_pt_root_id": post_synaptic_neurons},
                                    desired_resolution= [4,4,40],
                                    materialization_version=783)

syn_df = syn_df[syn_df['pre_pt_root_id'].isin(pre_synaptic_neurons)] # ensures that each input synapse is from one of the pre-synaptic neurons specified in pre_synaptic_neurons.

syn_df_filtered = syn_df.groupby('post_pt_root_id').filter(lambda x: len(x) >= 5) # add a filter equal or >5 syn

#syn_df_filtered.to_excel(r"C:\Users\mancinin\Documents\CAVEclient\CAVEclient\output_synapses_filtered.xlsx", index=False)

In [ ]:
syn_df_filtered

In [ ]:
# Build a NG link that has both pre-syn & post-syn line annotations with pre-syn segments
from IPython.display import HTML
#from nglui import statebuilder
#from neuroglancer import statebuilder
import nglui.statebuilder as statebuilder

img_layer = statebuilder.ImageLayerConfig(client.info.image_source())
seg_layer = statebuilder.SegmentationLayerConfig(client.info.segmentation_source(),
                                                selected_ids_column='post_pt_root_id')

# Build point mapper and annotation layer
lines = statebuilder.LineMapper(point_column_a='post_pt_position', 
                                point_column_b='post_pt_position')

anno_layer = statebuilder.AnnotationLayerConfig(name='Outsyn',
                                   mapping_rules=lines)

sb1 = statebuilder.StateBuilder([img_layer, seg_layer, anno_layer], resolution=[4,4,40])
sb1.render_state(syn_df_filtered, return_as='html')